In [ ]:
!pip -q install "transformers==4.18.0" "huggingface-hub"

  Could not find a version that satisfies the requirement transformers==5 (from versions: 0.1, 2.0.0, 2.1.0, 2.1.1, 2.2.0, 2.2.1, 2.2.2, 2.3.0, 2.4.0, 2.4.1, 2.5.0, 2.5.1, 2.6.0, 2.7.0, 2.8.0, 2.9.0, 2.9.1, 2.10.0, 2.11.0, 3.0.0, 3.0.1, 3.0.2, 3.1.0, 3.2.0, 3.3.0, 3.3.1, 3.4.0, 3.5.0, 3.5.1, 4.0.0rc1, 4.0.0, 4.0.1, 4.1.0, 4.1.1, 4.2.0, 4.2.1, 4.2.2, 4.3.0rc1, 4.3.0, 4.3.1, 4.3.2, 4.3.3, 4.4.0, 4.4.1, 4.4.2, 4.5.0, 4.5.1, 4.6.0, 4.6.1, 4.7.0, 4.8.0, 4.8.1, 4.8.2, 4.9.0, 4.9.1, 4.9.2, 4.10.0, 4.10.1, 4.10.2, 4.10.3, 4.11.0, 4.11.1, 4.11.2, 4.11.3, 4.12.0, 4.12.1, 4.12.2, 4.12.3, 4.12.4, 4.12.5, 4.13.0, 4.14.0, 4.14.1, 4.15.0, 4.16.0, 4.16.1, 4.16.2, 4.17.0, 4.18.0)
No matching distribution found for transformers==5


In [ ]:
import transformers; print(transformers.__version__)

import huggingface_hub; print(huggingface_hub.__version__)

SyntaxError: invalid syntax (2438148180.py, line 3)

In [4]:
from transformers import AutoTokenizer, AutoModelForMaskedLM
tokenizer = AutoTokenizer.from_pretrained("Qualcomm-AI-Research/BamiBERT")
bamibert = AutoModelForMaskedLM.from_pretrained("Qualcomm-AI-Research/BamiBERT")

TypeError: argument 'vocab': 'dict' object cannot be converted to 'Sequence'

In [ ]:
!pip install torch
import torch

In [ ]:
import torch; print(torch.__version__)

2.11.0+cpu


In [ ]:
text = f"bạn là {tokenizer.mask_token} à?"
input_ids = tokenizer.encode(text, return_tensors="pt")

with torch.no_grad():
    outputs = bamibert(input_ids)

logits = outputs.logits  # [1, seq_len, vocab_size] — decode được
mask_index = (input_ids[0] == tokenizer.mask_token_id).nonzero(as_tuple=True)[0]

predicted_id = logits[0, mask_index].argmax(dim=-1)
print(tokenizer.decode(predicted_id))

ớ


In [ ]:
from transformers import AutoModelForSequenceClassification

In [ ]:

model = AutoModelForSequenceClassification.from_pretrained(
    "Qualcomm-AI-Research/BamiBERT",
    num_labels=3,
    id2label={0: "tiêu cực", 1: "trung lập", 2: "tích cực"},
    label2id={"tiêu cực": 0, "trung lập": 1, "tích cực": 2},
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: Qualcomm-AI-Research/BamiBERT
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.decoder.bias       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
text = "như cứt"

inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits          # [1, num_labels]
predicted_class_id = logits.argmax(dim=-1).item()

print(predicted_class_id)

1


In [ ]:
import pandas as pd
from datasets import Dataset

df = pd.DataFrame({
    "text": [
        "Sản phẩm này dùng rất tốt, mình rất hài lòng!",
        "Giao hàng chậm, đóng gói cẩu thả, rất thất vọng",
        "Bình thường, không có gì đặc biệt",
        "Chất lượng tuyệt vời, sẽ mua lại lần sau",
        "Hàng lỗi, không dùng được, phí tiền",
        "Giá cả hợp lý, chất lượng tạm ổn",
    ],
    "label": [2, 0, 1, 2, 0, 1]
})

dataset = Dataset.from_pandas(df)

In [ ]:
dataset = dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = dataset["train"]
eval_dataset = dataset["test"]

In [ ]:
print(train_dataset)

Dataset({
    features: ['text', 'label'],
    num_rows: 4
})


In [ ]:
def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=128)

train_dataset = train_dataset.map(tokenize_fn, batched=True)
eval_dataset = eval_dataset.map(tokenize_fn, batched=True)

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

In [ ]:
from torch.optim import AdamW

In [ ]:
import torch.nn as nn

def calculate_loss(logits, labels):
  loss_fn = nn.MSELoss(reduction='none')
  cross_entropy_loss = loss_fn(torch.tensor(logits, dtype=torch.float32), torch.tensor(labels, dtype=torch.float32))
  return cross_entropy_loss

In [ ]:
a=torch.tensor(10, dtype=torch.float32)
print(a.shape)

torch.Size([])


In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./bamibert-cls",
    num_train_epochs=15,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=1,
    learning_rate=2e-5,

    logging_strategy="steps",
    logging_steps=1,          # print loss mỗi 1 step (để debug/xem cụ thể)
    logging_first_step=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,   # dataset đã tokenize
    eval_dataset=eval_dataset,
)

trainer.train()


Step,Training Loss
1,0.781931
2,0.700800
3,0.806765
4,0.616529
5,0.662042
6,0.556372
7,0.591252
8,0.592461
9,0.453605
10,0.673072


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=30, training_loss=0.5035216748714447, metrics={'train_runtime': 135.2769, 'train_samples_per_second': 0.444, 'train_steps_per_second': 0.222, 'total_flos': 3946701265920.0, 'train_loss': 0.5035216748714447, 'epoch': 15.0})

In [ ]:
trainer.evaluate()

{'eval_loss': 1.4964878559112549,
 'eval_runtime': 1.0715,
 'eval_samples_per_second': 1.866,
 'eval_steps_per_second': 1.866,
 'epoch': 15.0}

In [ ]:
optimizer = AdamW(model.parameters(), lr =5e-5)
for _ in range(10):
  for sample in train_dataset:
    input = tokenizer(sample["text"], return_tensors="pt", truncation=True, padding=True)
    out = model(**input,labels=torch.tensor(sample['label']))
    loss = out.loss
    loss.backward()
    optimizer.step
    optimizer.zero_grad()

    print(f"loss: {loss.item()}")

loss: 1.0966600179672241
loss: 0.9967914819717407
loss: 1.0580285787582397
loss: 1.140096664428711
loss: 1.0966600179672241
loss: 0.9967914819717407
loss: 1.0580285787582397
loss: 1.140096664428711
loss: 1.0966600179672241
loss: 0.9967914819717407
loss: 1.0580285787582397
loss: 1.140096664428711
loss: 1.0966600179672241
loss: 0.9967914819717407
loss: 1.0580285787582397
loss: 1.140096664428711
loss: 1.0966600179672241
loss: 0.9967914819717407
loss: 1.0580285787582397
loss: 1.140096664428711
loss: 1.0966600179672241
loss: 0.9967914819717407
loss: 1.0580285787582397
loss: 1.140096664428711
loss: 1.0966600179672241
loss: 0.9967914819717407
loss: 1.0580285787582397
loss: 1.140096664428711
loss: 1.0966600179672241
loss: 0.9967914819717407
loss: 1.0580285787582397
loss: 1.140096664428711
loss: 1.0966600179672241
loss: 0.9967914819717407
loss: 1.0580285787582397
loss: 1.140096664428711
loss: 1.0966600179672241
loss: 0.9967914819717407
loss: 1.0580285787582397
loss: 1.140096664428711


In [ ]:

model1 = AutoModelForSequenceClassification.from_pretrained(
    "/content/bamibert-cls/checkpoint-30",
    num_labels=3,
    id2label={0: "tiêu cực", 1: "trung lập", 2: "tích cực"},
    label2id={"tiêu cực": 0, "trung lập": 1, "tích cực": 2},
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
text = "sản phẩm chất lượng cao đó tôi thích"

inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

with torch.no_grad():
    outputs = model1(**inputs)

logits = outputs.logits          # [1, num_labels]
predicted_class_id = logits.argmax(dim=-1).item()

print(predicted_class_id)

2
